In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re

#Project paths
PROJECT_ROOT = Path(r"D:\Projects\AgriRisk and ROI Prediction")
DATA_DIR = PROJECT_ROOT / "Dump" / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

UPAG_RAW_DIR = RAW_DIR / "upag"
UPAG_PROCESSED_DIR = PROCESSED_DIR / "crop_yield"

UPAG_RAW_DIR.mkdir(parents=True, exist_ok=True)
UPAG_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("UPAg raw:", UPAG_RAW_DIR)
print("Processed:", UPAG_PROCESSED_DIR)

Project root: D:\Projects\AgriRisk and ROI Prediction
UPAg raw: D:\Projects\AgriRisk and ROI Prediction\Dump\data\raw\upag
Processed: D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield


In [2]:
#Inspecting current upag files
print("Existing UPAg RAW files:\n")

for f in UPAG_RAW_DIR.rglob("*"):
    if f.is_file():
        print(f.relative_to(UPAG_RAW_DIR))

Existing UPAg RAW files:

rice_upag_raw.csv
wheat_upag_raw.csv
maize_upag_raw.csv
urad_upag_raw.csv


In [3]:
#Loading all existing upag crops automatically
upag_files = list(UPAG_RAW_DIR.glob("*_upag_raw.csv"))

print("UPAg crop files found:", len(upag_files))

upag_data = {}

for file in upag_files:
    crop_name = file.stem.replace("_upag_raw", "").title()

    df = pd.read_csv(file)

    upag_data[crop_name] = df

    print(
        f"{crop_name:15s} | "
        f"Rows: {len(df):8,d} | "
        f"Columns: {len(df.columns):3d}"
    )

UPAg crop files found: 4
Rice            | Rows:      782 | Columns:  18
Wheat           | Rows:      782 | Columns:  18
Maize           | Rows:      782 | Columns:  18
Urad            | Rows:      782 | Columns:  18


In [4]:
#Which crops we currently have
print("\nCurrent crops:")

for crop in sorted(upag_data.keys()):
    print("-", crop)


Current crops:
- Maize
- Rice
- Urad
- Wheat


In [ ]:
#Inspecting crop names inside the data
for crop, df in upag_data.items():

    print(crop)

    possible_crop_columns = [
        c for c in df.columns
        if "crop" in c.lower()
    ]

    print("Possible crop columns:", possible_crop_columns)

    for col in possible_crop_columns:
        print(f"\nColumn: {col}")
        print(df[col].dropna().astype(str).unique()[:30])


Rice
Possible crop columns: []

Wheat
Possible crop columns: []

Maize
Possible crop columns: []

Urad
Possible crop columns: []


In [ ]:
#Inspecting all schemas
for crop, df in upag_data.items():

    print(crop)

    print(df.columns.tolist())


Rice
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'ricearea25', 'ricearea24', 'ricearea23', 'riceprod25', 'riceprod24', 'riceprod23', 'riceyld25', 'riceyld24', 'riceyld23', 'st_area(shape)', 'st_perimeter(shape)']

Wheat
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'wheatarea25', 'wheatarea24', 'wheatarea23', 'wheatprod25', 'wheatprod24', 'wheatprod23', 'wheatyld25', 'wheatyld24', 'wheatyld23', 'st_area(shape)', 'st_perimeter(shape)']

Maize
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'mazarea25', 'mazarea24', 'mazarea23', 'mazprod25', 'mazprod24', 'mazprod23', 'mazyld25', 'mazyld24', 'mazyld23', 'st_area(shape)', 'st_perimeter(shape)']

Urad
['objectid', 'uid', 'district', 'state', 'lgd_distcode', 'lgd_statecode', 'cencode2011', 'uradarea25', 'uradarea24', 'uradarea23', 'uradprod25', 'uradprod24', 'uradprod23', 'uradyld25', 'uradyld24', 'uradyld23', 'st

In [7]:
#Defining the expanded crop list
TARGET_CROPS = [
    "Rice",
    "Wheat",
    "Maize",
    "Urad",
    "Arhar/Tur",
    "Bajra",
    "Jowar",
    "Gram",
    "Moong",
    "Soybean",
    "Groundnut",
    "Rapeseed & Mustard"
]

print("Target crop count:", len(TARGET_CROPS))

for crop in TARGET_CROPS:
    print("-", crop)

Target crop count: 12
- Rice
- Wheat
- Maize
- Urad
- Arhar/Tur
- Bajra
- Jowar
- Gram
- Moong
- Soybean
- Groundnut
- Rapeseed & Mustard


In [ ]:
#Checking whether these crops already exist somewhere
print("Searching existing UPAg data for target crops...\n")

for crop_name, df in upag_data.items():

    print("File crop:", crop_name)

    for col in df.columns:

        if "crop" in col.lower():

            values = (
                df[col]
                .dropna()
                .astype(str)
                .str.strip()
                .unique()
            )

            matches = [
                value for value in values
                if any(
                    target.lower() in value.lower()
                    for target in TARGET_CROPS
                )
            ]

            if matches:
                print(f"{col}:")
                for value in matches[:50]:
                    print("  ", value)

Searching existing UPAg data for target crops...

File crop: Rice
File crop: Wheat
File crop: Maize
File crop: Urad


In [ ]:
#Inspecting year-suffix columns
for crop, df in upag_data.items():

    print(crop)

    # Identify area / production / yield columns
    relevant_cols = [
        c for c in df.columns
        if any(x in c.lower() for x in ["area", "prod", "yld"])
    ]

    print("\nRelevant columns:")
    print(relevant_cols)

    print("\nFirst 5 rows:")
    display(
        df[
            ["district", "state"] + relevant_cols
        ].head()
    )


Rice

Relevant columns:
['ricearea25', 'ricearea24', 'ricearea23', 'riceprod25', 'riceprod24', 'riceprod23', 'riceyld25', 'riceyld24', 'riceyld23', 'st_area(shape)']

First 5 rows:


,district,state,ricearea25,ricearea24,ricearea23,riceprod25,riceprod24,riceprod23,riceyld25,riceyld24,riceyld23,st_area(shape)
0,Dakshina Kannada,Karnataka,0.04,0.05,0.07,0.10,0.16,0.21,2848.0,3035.0,2928.0,5.131938e+09
1,Udupi,Karnataka,0.28,0.35,0.40,0.79,1.04,1.17,2807.0,2993.0,2894.0,3.806404e+09
2,Uttara Kannada,Karnataka,0.38,0.38,0.39,0.83,0.73,0.82,2210.0,1917.0,2104.0,1.108870e+10
3,Bahraich,Uttar Pradesh,1.66,1.60,1.71,5.09,4.59,3.98,3066.0,2864.0,2328.0,6.287853e+09
4,Balrampur,Uttar Pradesh,1.21,1.27,1.35,3.32,3.78,2.87,2743.0,2972.0,2133.0,4.263626e+09



Wheat

Relevant columns:
['wheatarea25', 'wheatarea24', 'wheatarea23', 'wheatprod25', 'wheatprod24', 'wheatprod23', 'wheatyld25', 'wheatyld24', 'wheatyld23', 'st_area(shape)']

First 5 rows:


,district,state,wheatarea25,wheatarea24,wheatarea23,wheatprod25,wheatprod24,wheatprod23,wheatyld25,wheatyld24,wheatyld23,st_area(shape)
0,Dakshina Kannada,Karnataka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.131938e+09
1,Udupi,Karnataka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.806404e+09
2,Uttara Kannada,Karnataka,0.00,NaN,NaN,0.00,NaN,NaN,1255.0,NaN,NaN,1.108870e+10
3,Bahraich,Uttar Pradesh,1.90,1.86,1.88,6.69,5.89,6.08,3528.0,3168.0,3242.0,6.287853e+09
4,Balrampur,Uttar Pradesh,1.02,1.06,1.05,3.21,3.53,3.21,3135.0,3329.0,3049.0,4.263626e+09



Maize

Relevant columns:
['mazarea25', 'mazarea24', 'mazarea23', 'mazprod25', 'mazprod24', 'mazprod23', 'mazyld25', 'mazyld24', 'mazyld23', 'st_area(shape)']

First 5 rows:


,district,state,mazarea25,mazarea24,mazarea23,mazprod25,mazprod24,mazprod23,mazyld25,mazyld24,mazyld23,st_area(shape)
0,Dakshina Kannada,Karnataka,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.131938e+09
1,Udupi,Karnataka,0.00,0.00,0.00,0.00,0.00,0.00,3572.0,3884.0,3556.0,3.806404e+09
2,Uttara Kannada,Karnataka,0.13,0.10,0.10,0.49,0.44,0.41,3674.0,4417.0,3960.0,1.108870e+10
3,Bahraich,Uttar Pradesh,0.51,1.00,0.95,1.01,1.78,1.47,1999.0,1781.0,1548.0,6.287853e+09
4,Balrampur,Uttar Pradesh,0.03,0.04,0.04,0.07,0.06,0.07,2414.0,1728.0,1764.0,4.263626e+09



Urad

Relevant columns:
['uradarea25', 'uradarea24', 'uradarea23', 'uradprod25', 'uradprod24', 'uradprod23', 'uradyld25', 'uradyld24', 'uradyld23', 'st_area(shape)']

First 5 rows:


,district,state,uradarea25,uradarea24,uradarea23,uradprod25,uradprod24,uradprod23,uradyld25,uradyld24,uradyld23,st_area(shape)
0,Dakshina Kannada,Karnataka,NaN,0.00,NaN,NaN,0.00,NaN,NaN,0.0,NaN,5.131938e+09
1,Udupi,Karnataka,0.00,0.02,0.03,0.00,0.01,0.01,485.0,563.0,531.0,3.806404e+09
2,Uttara Kannada,Karnataka,0.00,0.00,0.00,0.00,0.00,0.00,430.0,557.0,522.0,1.108870e+10
3,Bahraich,Uttar Pradesh,0.03,0.03,0.02,0.02,0.02,0.02,767.0,724.0,616.0,6.287853e+09
4,Balrampur,Uttar Pradesh,0.01,0.02,0.01,0.01,0.02,0.01,448.0,628.0,506.0,4.263626e+09


In [ ]:
#Checking the metadata columns
for crop, df in upag_data.items():

    print(crop)

    print("\nDistrict count:", df["district"].nunique())
    print("State count:", df["state"].nunique())

    print("\nSample states:")
    print(df["state"].drop_duplicates().head(15).tolist())

    print("\nSample districts:")
    print(df["district"].drop_duplicates().head(15).tolist())

    print("\nLGD state codes:")
    print(df["lgd_statecode"].drop_duplicates().head(20).tolist())

    print("\nLGD district codes:")
    print(df["lgd_distcode"].drop_duplicates().head(20).tolist())


Rice

District count: 779
State count: 36

Sample states:
['Karnataka', 'Uttar Pradesh', 'West Bengal', 'Andhra Pradesh', 'Tamil Nadu', 'Puducherry', 'Odisha', 'Sikkim', 'Uttarakhand', 'Punjab', 'The Dadra And Nagar Haveli And Daman And Diu', 'Maharashtra', 'Goa', 'Lakshadweep', 'Kerala']

Sample districts:
['Dakshina Kannada', 'Udupi', 'Uttara Kannada', 'Bahraich', 'Balrampur', 'Kheri', 'Mahrajganj', 'Pilibhit', 'Shrawasti', 'Siddharthnagar', 'North 24 Parganas', 'South 24 Parganas', 'Cooch Behar', 'Darjeeling', 'Dakshin Dinajpur']

LGD state codes:
[29, 9, 19, 28, 33, 34, 21, 11, 5, 3, 38, 27, 30, 31, 32, 35, 17, 15, 16, 12]

LGD district codes:
[534, 549, 550, 125, 127, 159, 164, 173, 181, 182, 303, 304, 308, 309, 310, 311, 313, 314, 316, 317]

Wheat

District count: 779
State count: 36

Sample states:
['Karnataka', 'Uttar Pradesh', 'West Bengal', 'Andhra Pradesh', 'Tamil Nadu', 'Puducherry', 'Odisha', 'Sikkim', 'Uttarakhand', 'Punjab', 'The Dadra And Nagar Haveli And Daman And Diu

In [ ]:
#Checking missing values
for crop, df in upag_data.items():

    print(crop)

    relevant_cols = [
        c for c in df.columns
        if any(x in c.lower() for x in ["area", "prod", "yld"])
    ]

    missing = df[relevant_cols].isna().sum()

    print(missing)


Rice
ricearea25        176
ricearea24         87
ricearea23        107
riceprod25        176
riceprod24         87
riceprod23        107
riceyld25         176
riceyld24          87
riceyld23         107
st_area(shape)      0
dtype: int64

Wheat
wheatarea25       300
wheatarea24       243
wheatarea23       229
wheatprod25       300
wheatprod24       248
wheatprod23       229
wheatyld25        300
wheatyld24        248
wheatyld23        229
st_area(shape)      0
dtype: int64

Maize
mazarea25         173
mazarea24          89
mazarea23         127
mazprod25         174
mazprod24          89
mazprod23         127
mazyld25          174
mazyld24           89
mazyld23          127
st_area(shape)      0
dtype: int64

Urad
uradarea25        275
uradarea24        186
uradarea23        201
uradprod25        275
uradprod24        187
uradprod23        201
uradyld25         275
uradyld24         187
uradyld23         201
st_area(shape)      0
dtype: int64


In [ ]:
#Checking zero values
for crop, df in upag_data.items():

    print(crop)

    relevant_cols = [
        c for c in df.columns
        if any(x in c.lower() for x in ["area", "prod", "yld"])
    ]

    for col in relevant_cols:
        zero_count = (df[col] == 0).sum()

        print(
            f"{col:20s} "
            f"zeros = {zero_count:4d} "
            f"({zero_count / len(df) * 100:.2f}%)"
        )


Rice
ricearea25           zeros =   25 (3.20%)
ricearea24           zeros =   43 (5.50%)
ricearea23           zeros =   37 (4.73%)
riceprod25           zeros =   29 (3.71%)
riceprod24           zeros =   37 (4.73%)
riceprod23           zeros =   30 (3.84%)
riceyld25            zeros =    0 (0.00%)
riceyld24            zeros =    0 (0.00%)
riceyld23            zeros =    0 (0.00%)
st_area(shape)       zeros =    0 (0.00%)

Wheat
wheatarea25          zeros =   91 (11.64%)
wheatarea24          zeros =  128 (16.37%)
wheatarea23          zeros =  138 (17.65%)
wheatprod25          zeros =   83 (10.61%)
wheatprod24          zeros =   99 (12.66%)
wheatprod23          zeros =  113 (14.45%)
wheatyld25           zeros =    1 (0.13%)
wheatyld24           zeros =    0 (0.00%)
wheatyld23           zeros =    0 (0.00%)
st_area(shape)       zeros =    0 (0.00%)

Maize
mazarea25            zeros =  149 (19.05%)
mazarea24            zeros =  180 (23.02%)
mazarea23            zeros =  159 (20.33%)
mazpr

In [ ]:
#Seaching project for remaining crops
from pathlib import Path
import pandas as pd

#Project root
PROJECT_ROOT = Path.cwd().parent.parent

print("Project root:")
print(PROJECT_ROOT)

#Crops we already have
existing_crops = {
    "Rice",
    "Wheat",
    "Maize",
    "Urad"
}

#Remaining crops required
remaining_crops = {
    "Arhar": [
        "arhar",
        "tur",
        "pigeonpea",
        "pigeon_pea"
    ],
    
    "Bajra": [
        "bajra",
        "pearlmillet",
        "pearl_millet"
    ],
    
    "Jowar": [
        "jowar",
        "sorghum"
    ],
    
    "Gram": [
        "gram",
        "chickpea",
        "chana"
    ],
    
    "Moong": [
        "moong",
        "mung",
        "greengram",
        "green_gram"
    ],
    
    "Soybean": [
        "soybean",
        "soyabean"
    ],
    
    "Groundnut": [
        "groundnut",
        "peanut"
    ],
    
    "Rapeseed & Mustard": [
        "rapeseed",
        "mustard",
        "rapeseed_mustard",
        "rapeseed_and_mustard"
    ]
}

#Searching all csv files
csv_files = list(PROJECT_ROOT.rglob("*.csv"))

print("\nTotal CSV files found:", len(csv_files))

print("SEARCHING FOR REMAINING CROPS")

results = []

for crop, keywords in remaining_crops.items():

    print("TARGET CROP:", crop)

    crop_found = False

    for file_path in csv_files:

        filename = file_path.name.lower()

        # Search filename
        filename_match = any(
            keyword.lower() in filename
            for keyword in keywords
        )

        # Search path
        path_match = any(
            keyword.lower() in str(file_path).lower()
            for keyword in keywords
        )

        if filename_match or path_match:

            crop_found = True

            print("FOUND:")
            print(file_path)

            results.append({
                "target_crop": crop,
                "file": str(file_path),
                "match_type": "filename/path"
            })

    if not crop_found:
        print("No filename/path match found.")


results_df = pd.DataFrame(results)

print("SEARCH SUMMARY")

if len(results_df) > 0:
    display(results_df)
else:
    print("No files matched the remaining crop names.")

Project root:
d:\Projects\AgriRisk and ROI Prediction

Total CSV files found: 30

SEARCHING FOR REMAINING CROPS

--------------------------------------------------------------------------------
TARGET CROP: Arhar
--------------------------------------------------------------------------------
FOUND:
d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\arhar_des_1997_98_2022_23.csv

--------------------------------------------------------------------------------
TARGET CROP: Bajra
--------------------------------------------------------------------------------
FOUND:
d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\bajra_des_1997_98_2022_23.csv

--------------------------------------------------------------------------------
TARGET CROP: Jowar
--------------------------------------------------------------------------------
FOUND:
d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\jowar_des_1997_98_2022_23.csv

-----------------

,target_crop,file,match_type
0,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
1,Bajra,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
2,Jowar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
3,Gram,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
4,Moong,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
5,Soybean,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
6,Groundnut,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path
7,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,filename/path


In [ ]:
#Searching csv column names
print("SEARCHING CSV COLUMN NAMES FOR REMAINING CROPS")

column_results = []

for file_path in csv_files:

    try:
        # Read only header
        header = pd.read_csv(
            file_path,
            nrows=0
        )

        columns = [str(c).lower() for c in header.columns]

        for crop, keywords in remaining_crops.items():

            matched_columns = []

            for col in columns:

                if any(
                    keyword.lower() in col
                    for keyword in keywords
                ):
                    matched_columns.append(col)

            if matched_columns:

                column_results.append({
                    "target_crop": crop,
                    "file": str(file_path),
                    "matched_columns": matched_columns
                })

    except Exception as e:
        print("Could not read:", file_path)
        print("Reason:", e)

if column_results:

    column_results_df = pd.DataFrame(column_results)

    display(column_results_df)

else:

    print("No remaining crop names found in CSV column names.")

SEARCHING CSV COLUMN NAMES FOR REMAINING CROPS
No remaining crop names found in CSV column names.


In [ ]:
#Searching actual data values
print("SEARCHING CROP VALUES INSIDE CSV FILES")

value_results = []

all_crop_keywords = {}

for crop, keywords in remaining_crops.items():
    all_crop_keywords[crop] = [
        k.lower().replace("_", " ")
        for k in keywords
    ]

for file_path in csv_files:

    try:

        # Read a sample first
        sample = pd.read_csv(
            file_path,
            nrows=1000,
            low_memory=False
        )

        # Look for likely crop columns
        possible_crop_columns = [
            col for col in sample.columns
            if any(
                word in str(col).lower()
                for word in [
                    "crop",
                    "commodity",
                    "cereal",
                    "pulse",
                    "crop_name"
                ]
            )
        ]

        if not possible_crop_columns:
            continue

        for col in possible_crop_columns:

            values = (
                sample[col]
                .dropna()
                .astype(str)
                .str.lower()
                .str.strip()
            )

            for crop, keywords in all_crop_keywords.items():

                for keyword in keywords:

                    matches = values[
                        values.str.contains(
                            keyword,
                            regex=False,
                            na=False
                        )
                    ]

                    if len(matches) > 0:

                        value_results.append({
                            "target_crop": crop,
                            "file": str(file_path),
                            "column": col,
                            "matched_keyword": keyword,
                            "sample_matches": matches.unique()[:10].tolist()
                        })

    except Exception as e:
        print("Skipped:", file_path)
        print("Reason:", e)


if value_results:

    value_results_df = pd.DataFrame(value_results)

    display(value_results_df)

else:

    print("No remaining crops found as crop values.")

SEARCHING CROP VALUES INSIDE CSV FILES


,target_crop,file,column,matched_keyword,sample_matches
0,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_name,arhar,[arhar/tur]
1,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_name,tur,"[arhar/tur, turmeric]"
2,Bajra,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_name,bajra,[bajra]
3,Jowar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_name,jowar,[jowar]
4,Gram,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_name,gram,"[gram, horse-gram, moong(green gram)]"
...,...,...,...,...,...
67,Groundnut,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop,groundnut,[groundnut]
68,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_source_name,rapeseed,[rapeseed &mustard]
69,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_source_name,mustard,[rapeseed &mustard]
70,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop,rapeseed,[rapeseed & mustard]


In [ ]:
#Final crop availability report
already_available = {"Maize", "Rice", "Urad", "Wheat"}

remaining_crops = [
    "Arhar",
    "Bajra",
    "Jowar",
    "Gram",
    "Moong",
    "Soybean",
    "Groundnut",
    "Rapeseed & Mustard"
]

# Results from the crop-value search
value_found_crops = set()

if "value_results_df" in globals() and len(value_results_df) > 0:
    value_found_crops = set(
        value_results_df["target_crop"].dropna().unique()
    )

# Results from column-name search, if that dataframe exists
column_found_crops = set()

if "column_results_df" in globals() and len(column_results_df) > 0:
    column_found_crops = set(
        column_results_df["target_crop"].dropna().unique()
    )

# Combine both
found_crops = value_found_crops.union(column_found_crops)

print("CROP AVAILABILITY REPORT")

print("\nAlready available:")
for crop in sorted(already_available):
    print(f"  ✓ {crop}")

print("\nRemaining crops:")
for crop in remaining_crops:
    if crop in found_crops:
        print(f"  ✓ {crop}  -> FOUND IN EXISTING DATA")
    else:
        print(f"  ? {crop}  -> NOT YET VERIFIED")

print("SUMMARY")

print("Found remaining crops:")
for crop in remaining_crops:
    if crop in found_crops:
        print(f"  ✓ {crop}")

print("\nStill unresolved:")
for crop in remaining_crops:
    if crop not in found_crops:
        print(f"  ? {crop}")

CROP AVAILABILITY REPORT

Already available:
  ✓ Maize
  ✓ Rice
  ✓ Urad
  ✓ Wheat

Remaining crops:
  ✓ Arhar  -> FOUND IN EXISTING DATA
  ✓ Bajra  -> FOUND IN EXISTING DATA
  ✓ Jowar  -> FOUND IN EXISTING DATA
  ✓ Gram  -> FOUND IN EXISTING DATA
  ✓ Moong  -> FOUND IN EXISTING DATA
  ✓ Soybean  -> FOUND IN EXISTING DATA
  ✓ Groundnut  -> FOUND IN EXISTING DATA
  ✓ Rapeseed & Mustard  -> FOUND IN EXISTING DATA

SUMMARY
Found remaining crops:
  ✓ Arhar
  ✓ Bajra
  ✓ Jowar
  ✓ Gram
  ✓ Moong
  ✓ Soybean
  ✓ Groundnut
  ✓ Rapeseed & Mustard

Still unresolved:


In [ ]:
#Identifying exact source files for remaining crops
remaining_crops = {
    "Arhar": ["arhar", "tur"],
    "Bajra": ["bajra"],
    "Jowar": ["jowar"],
    "Gram": ["gram"],
    "Moong": ["moong(green gram)", "green gram"],
    "Soybean": ["soyabean", "soybean"],
    "Groundnut": ["groundnut"],
    "Rapeseed & Mustard": ["rapeseed &mustard", "rapeseed", "mustard"]
}

# Get CSV files again directly from the project
csv_files = list(PROJECT_ROOT.rglob("*.csv"))

print("EXACT SOURCE FILES FOR REMAINING CROPS")

source_results = []

for crop, keywords in remaining_crops.items():

    print(f"TARGET CROP: {crop}")

    crop_matches = []

    for path in csv_files:

        try:
            # Read only a small sample first
            sample = pd.read_csv(
                path,
                nrows=10,
                low_memory=False
            )

            # Check column names
            for col in sample.columns:

                # Only inspect text-like columns
                if sample[col].dtype == "object":

                    values = (
                        sample[col]
                        .dropna()
                        .astype(str)
                        .str.lower()
                    )

                    for keyword in keywords:

                        if values.str.contains(
                            keyword.lower(),
                            regex=False,
                            na=False
                        ).any():

                            crop_matches.append({
                                "crop": crop,
                                "file": str(path),
                                "column": col,
                                "keyword": keyword
                            })

                            break

        except Exception:
            continue

    # Remove duplicates
    unique_matches = []
    seen = set()

    for m in crop_matches:
        key = (m["file"], m["column"])

        if key not in seen:
            seen.add(key)
            unique_matches.append(m)

    if unique_matches:

        for m in unique_matches:
            print(
                f"✓ {m['file']} | "
                f"column={m['column']} | "
                f"match={m['keyword']}"
            )

        source_results.extend(unique_matches)

    else:
        print("✗ No source file identified in sample.")

# Store results
source_results_df = pd.DataFrame(source_results)

print("SOURCE FILE SEARCH COMPLETE")

if len(source_results_df) > 0:
    print(f"Total source matches: {len(source_results_df)}")
    display(source_results_df)
else:
    print("No source matches found.")

EXACT SOURCE FILES FOR REMAINING CROPS

--------------------------------------------------------------------------------
TARGET CROP: Arhar
--------------------------------------------------------------------------------
✓ d:\Projects\AgriRisk and ROI Prediction\Dump\data\raw\des\des_apy_raw.csv | column=crop_name | match=arhar
✓ d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\des_apy_2013_14_2022_23.csv | column=crop | match=arhar
✓ d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\des_apy_2013_14_2022_23.csv | column=crop_source_name | match=arhar
✓ d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\arhar_des_1997_98_2022_23.csv | column=crop | match=arhar
✓ d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\arhar_des_1997_98_2022_23.csv | column=keyword | match=arhar
✓ d:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\bajra_des_1997_98_2022_23.csv | column=crop | match=arhar
✓ d:

,crop,file,column,keyword
0,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_name,arhar
1,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop,arhar
2,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_source_name,arhar
3,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop,arhar
4,Arhar,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,keyword,arhar
...,...,...,...,...
82,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,original_names,rapeseed &mustard
83,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop,rapeseed
84,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,original_names,rapeseed &mustard
85,Rapeseed & Mustard,d:\Projects\AgriRisk and ROI Prediction\Dump\d...,crop_source_name,rapeseed &mustard


In [ ]:
#Loading raw des data
from pathlib import Path
import pandas as pd

print("LOADING DES RAW DATA")

# Project root
PROJECT_ROOT = Path.cwd().parent

if PROJECT_ROOT.name.lower() != "dump":
    possible_dump = PROJECT_ROOT / "Dump"
    if possible_dump.exists():
        PROJECT_ROOT = possible_dump

# Correct DES raw path
DES_RAW = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "des"
    / "des_apy_raw.csv"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nDES raw file:")
print(DES_RAW)

print("\nExists:", DES_RAW.exists())

if not DES_RAW.exists():
    raise FileNotFoundError(
        f"\nDES raw file not found at:\n{DES_RAW}\n"
    )

# Load data
des_raw = pd.read_csv(
    DES_RAW,
    low_memory=False
)

print("\nDES raw data loaded successfully.")
print("Shape:", des_raw.shape)

print("\nColumns:")
print(des_raw.columns.tolist())

print("\nFirst 5 rows:")
display(des_raw.head())

LOADING DES RAW DATA
Project root:
d:\Projects\AgriRisk and ROI Prediction\Dump

DES raw file:
d:\Projects\AgriRisk and ROI Prediction\Dump\data\raw\des\des_apy_raw.csv

Exists: True

DES raw data loaded successfully.
Shape: (455359, 17)

Columns:
['_id', 'id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit']

First 5 rows:


,_id,id,year,state_name,state_code,district_name,district_code,crop_name,crop_code,crop_type,season,area,area_unit,production,production_unit,yield,yield_unit
0,1,0,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Arhar/Tur,202.0,Pulses,Kharif,21400.0,Hectare,2600.0,Tonnes,0.121,Tonnes/Hectare
1,2,1,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Bajra,103.0,Cereals,Kharif,1400.0,Hectare,500.0,Tonnes,0.357,Tonnes/Hectare
2,3,2,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Castor Seed,1002.0,Oilseeds,Kharif,1000.0,Hectare,100.0,Tonnes,0.100,Tonnes/Hectare
3,4,3,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Cotton(Lint),1101.0,Fiber Crops,Kharif,7300.0,Hectare,9400.0,Tonnes,1.288,Tonnes/Hectare
4,5,4,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Dry Chillies,502.0,Spices,Kharif,3700.0,Hectare,7100.0,Tonnes,1.919,Tonnes/Hectare


In [ ]:
#Identifying and extracting remaining target crops
# Target crops required for crop expansion
TARGET_CROPS = {
    "Arhar": ["arhar/tur", "arhar", "tur"],
    "Bajra": ["bajra"],
    "Jowar": ["jowar"],
    "Gram": ["gram", "chickpea", "bengal gram"],
    "Moong": ["moong", "moong(green gram)", "green gram"],
    "Soybean": ["soyabean", "soybean"],
    "Groundnut": ["groundnut"],
    "Rapeseed & Mustard": [
        "rapeseed &mustard",
        "rapeseed & mustard",
        "rapeseed",
        "mustard"
    ]
}

print("EXTRACTING REMAINING TARGET CROPS FROM DES")

# Working on a copy
des = des_raw.copy()

# Normalizing crop names only for matching
des["_crop_name_normalized"] = (
    des["crop_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

crop_dfs = {}

for target_crop, aliases in TARGET_CROPS.items():

    aliases_normalized = [
        x.strip().lower()
        for x in aliases
    ]

    mask = des["_crop_name_normalized"].isin(
        aliases_normalized
    )

    crop_df = des.loc[mask].copy()

    # Storing standardized target crop name
    crop_df["crop"] = target_crop

    crop_dfs[target_crop] = crop_df

    print(f"{target_crop}")
    print("Rows found:", len(crop_df))

    if len(crop_df) > 0:
        print(
            "Original crop names:",
            sorted(crop_df["crop_name"].dropna().unique().tolist())
        )

        print(
            "Years:",
            crop_df["year"].min(),
            "to",
            crop_df["year"].max()
        )

        print(
            "States:",
            crop_df["state_name"].nunique()
        )

        print(
            "Districts:",
            crop_df["district_name"].nunique()
        )
    else:
        print("WARNING: No rows found.")


print("EXTRACTION COMPLETE")

EXTRACTING REMAINING TARGET CROPS FROM DES

----------------------------------------------------------------------
Arhar
----------------------------------------------------------------------
Rows found: 13230
Original crop names: ['Arhar/Tur']
Years: 1997-1998 to 2022-2023
States: 30
Districts: 669

----------------------------------------------------------------------
Bajra
----------------------------------------------------------------------
Rows found: 10376
Original crop names: ['Bajra']
Years: 1997-1998 to 2022-2023
States: 24
Districts: 517

----------------------------------------------------------------------
Jowar
----------------------------------------------------------------------
Rows found: 13841
Original crop names: ['Jowar']
Years: 1997-1998 to 2022-2023
States: 24
Districts: 532

----------------------------------------------------------------------
Gram
----------------------------------------------------------------------
Rows found: 12119
Original crop names: ['Gr

In [ ]:
#Verifying all extracted crops
print("EXTRACTED CROP SUMMARY")

summary = []

for crop, df in crop_dfs.items():

    summary.append({
        "crop": crop,
        "rows": len(df),
        "states": df["state_name"].nunique(),
        "districts": df["district_name"].nunique(),
        "min_year": df["year"].min() if len(df) else None,
        "max_year": df["year"].max() if len(df) else None,
        "original_names": ", ".join(
            sorted(df["crop_name"].dropna().unique().astype(str))
        ) if len(df) else ""
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

EXTRACTED CROP SUMMARY


,crop,rows,states,districts,min_year,max_year,original_names
0,Arhar,13230,30,669,1997-1998,2022-2023,Arhar/Tur
1,Bajra,10376,24,517,1997-1998,2022-2023,Bajra
2,Jowar,13841,24,532,1997-1998,2022-2023,Jowar
3,Gram,12119,29,643,1997-1998,2022-2023,Gram
4,Moong,23419,30,685,1997-1998,2022-2023,Moong(Green Gram)
5,Soybean,6160,25,452,1997-1998,2022-2023,Soyabean
6,Groundnut,18623,29,600,1997-1998,2022-2023,Groundnut
7,Rapeseed & Mustard,12661,33,662,1997-1998,2022-2023,Rapeseed &Mustard


In [ ]:
#Standardizing remaining crop data
print("STANDARDIZING REMAINING DES CROP DATA")

standardized_crops = {}

for crop, df in crop_dfs.items():

    temp = df.copy()

    # Keeping only the fields required for crop-level harmonization
    temp = temp[
        [
            "year",
            "state_name",
            "state_code",
            "district_name",
            "district_code",
            "crop_name",
            "crop_code",
            "crop_type",
            "season",
            "area",
            "area_unit",
            "production",
            "production_unit",
            "yield",
            "yield_unit",
            "crop"
        ]
    ].copy()

    # Standard column names
    temp = temp.rename(columns={
        "year": "year",
        "state_name": "state",
        "state_code": "state_code",
        "district_name": "district",
        "district_code": "district_code",
        "crop_name": "crop_source_name",
        "crop_code": "crop_code",
        "crop_type": "crop_type",
        "season": "season",
        "area": "area_ha",
        "area_unit": "area_unit",
        "production": "production_tonnes",
        "production_unit": "production_unit",
        "yield": "yield_reported",
        "yield_unit": "yield_unit",
        "crop": "crop"
    })

    # Converting numeric fields explicitly
    numeric_columns = [
        "state_code",
        "district_code",
        "crop_code",
        "area_ha",
        "production_tonnes",
        "yield_reported"
    ]

    for col in numeric_columns:
        temp[col] = pd.to_numeric(
            temp[col],
            errors="coerce"
        )

    # Cleaning string columns
    string_columns = [
        "state",
        "district",
        "crop_source_name",
        "crop_type",
        "season",
        "area_unit",
        "production_unit",
        "yield_unit",
        "crop"
    ]

    for col in string_columns:
        temp[col] = temp[col].astype("string").str.strip()

    standardized_crops[crop] = temp

    print(
        f"{crop:22s} -> "
        f"rows={len(temp):6d}, "
        f"columns={len(temp.columns):2d}"
    )

print("STANDARDIZATION COMPLETE")

STANDARDIZING REMAINING DES CROP DATA
Arhar                  -> rows= 13230, columns=16
Bajra                  -> rows= 10376, columns=16
Jowar                  -> rows= 13841, columns=16
Gram                   -> rows= 12119, columns=16
Moong                  -> rows= 23419, columns=16
Soybean                -> rows=  6160, columns=16
Groundnut              -> rows= 18623, columns=16
Rapeseed & Mustard     -> rows= 12661, columns=16

STANDARDIZATION COMPLETE


In [ ]:
#Validating standardized crop data
print("VALIDATING STANDARDIZED DATA")

validation_rows = []

for crop, df in standardized_crops.items():

    validation_rows.append({
        "crop": crop,
        "rows": len(df),
        "unique_states": df["state"].nunique(),
        "unique_districts": df["district"].nunique(),
        "unique_years": df["year"].nunique(),
        "missing_area": df["area_ha"].isna().sum(),
        "missing_production": df["production_tonnes"].isna().sum(),
        "missing_yield": df["yield_reported"].isna().sum(),
        "zero_area": (df["area_ha"] == 0).sum(),
        "zero_production": (df["production_tonnes"] == 0).sum()
    })

standardized_validation_df = pd.DataFrame(validation_rows)

display(standardized_validation_df)

print("\nSample standardized Arhar data:")
display(
    standardized_crops["Arhar"].head()
)

VALIDATING STANDARDIZED DATA


,crop,rows,unique_states,unique_districts,unique_years,missing_area,missing_production,missing_yield,zero_area,zero_production
0,Arhar,13230,30,669,26,0,121,0,0,25
1,Bajra,10376,24,517,26,0,56,0,0,25
2,Jowar,13841,24,532,26,0,87,0,0,37
3,Gram,12119,29,643,26,0,144,0,0,47
4,Moong,23419,30,685,26,0,289,0,0,113
5,Soybean,6160,25,452,26,0,35,0,0,6
6,Groundnut,18623,29,600,26,0,71,0,0,44
7,Rapeseed & Mustard,12661,33,662,26,0,115,0,0,30



Sample standardized Arhar data:


,year,state,state_code,district,district_code,crop_source_name,crop_code,crop_type,season,area_ha,area_unit,production_tonnes,production_unit,yield_reported,yield_unit,crop
0,1997-1998,Andhra Pradesh,28,Ananthapuramu,502,Arhar/Tur,202.0,Pulses,Kharif,21400.0,Hectare,2600.0,Tonnes,0.121,Tonnes/Hectare,Arhar
37,1997-1998,Andhra Pradesh,28,Chittoor,503,Arhar/Tur,202.0,Pulses,Kharif,6100.0,Hectare,900.0,Tonnes,0.148,Tonnes/Hectare,Arhar
64,1997-1998,Andhra Pradesh,28,East Godavari,505,Arhar/Tur,202.0,Pulses,Kharif,1600.0,Hectare,300.0,Tonnes,0.188,Tonnes/Hectare,Arhar
96,1997-1998,Andhra Pradesh,28,Guntur,506,Arhar/Tur,202.0,Pulses,Kharif,28400.0,Hectare,14900.0,Tonnes,0.525,Tonnes/Hectare,Arhar
97,1997-1998,Andhra Pradesh,28,Guntur,506,Arhar/Tur,202.0,Pulses,Rabi,1800.0,Hectare,1000.0,Tonnes,0.556,Tonnes/Hectare,Arhar


In [ ]:
#Finding standardized crop dataframes
import pandas as pd
from pathlib import Path

print("SEARCHING FOR STANDARDIZED CROP DATAFRAMES")

target_crops = [
    "Arhar",
    "Bajra",
    "Jowar",
    "Gram",
    "Moong",
    "Soybean",
    "Groundnut",
    "Rapeseed & Mustard"
]

# Taking a snapshot first so globals() does not change during iteration
global_items = list(globals().items())

crop_dfs = {}

for crop in target_crops:
    candidates = []

    for name, obj in global_items:
        if isinstance(obj, pd.DataFrame):
            # Checking whether dataframe has the standardized crop column
            if "crop" in obj.columns:
                try:
                    values = (
                        obj["crop"]
                        .dropna()
                        .astype(str)
                        .str.strip()
                    )

                    if crop in values.unique():
                        candidates.append((name, obj))
                except Exception:
                    pass

    print(f"\n{crop}")
    print("-" * 60)

    if candidates:
        # Prefering the largest matching dataframe
        name, df = max(candidates, key=lambda x: len(x[1]))

        crop_dfs[crop] = df.copy()

        print(f"✓ Found DataFrame: {name}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")
    else:
        print("✗ No matching DataFrame found")

print("FOUND CROPS")

for crop, df in crop_dfs.items():
    print(f"{crop:22s} -> {df.shape}")

SEARCHING FOR STANDARDIZED CROP DATAFRAMES

Arhar
------------------------------------------------------------
✓ Found DataFrame: source_results_df
  Shape: (87, 4)
  Columns: ['crop', 'file', 'column', 'keyword']

Bajra
------------------------------------------------------------
✓ Found DataFrame: source_results_df
  Shape: (87, 4)
  Columns: ['crop', 'file', 'column', 'keyword']

Jowar
------------------------------------------------------------
✓ Found DataFrame: source_results_df
  Shape: (87, 4)
  Columns: ['crop', 'file', 'column', 'keyword']

Gram
------------------------------------------------------------
✓ Found DataFrame: source_results_df
  Shape: (87, 4)
  Columns: ['crop', 'file', 'column', 'keyword']

Moong
------------------------------------------------------------
✓ Found DataFrame: source_results_df
  Shape: (87, 4)
  Columns: ['crop', 'file', 'column', 'keyword']

Soybean
------------------------------------------------------------
✓ Found DataFrame: source_results

In [ ]:
#Saving the 8 standardized des datasets
PROJECT_ROOT = Path.cwd().parents[1]

# Safety check: locate the actual project root
possible_roots = [
    Path.cwd().parents[1],
    Path.cwd().parents[2],
    Path(r"D:\Projects\AgriRisk and ROI Prediction\Dump")
]

for root in possible_roots:
    if (root / "data").exists():
        PROJECT_ROOT = root
        break

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "crop_yield"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("SAVING STANDARDIZED CROP DATA")

print("Project root:")
print(PROJECT_ROOT)

print("\nOutput directory:")
print(PROCESSED_DIR)

saved_files = {}

for crop, df in crop_dfs.items():

    # Creating safe filename
    filename_map = {
        "Arhar": "arhar_des_1997_98_2022_23.csv",
        "Bajra": "bajra_des_1997_98_2022_23.csv",
        "Jowar": "jowar_des_1997_98_2022_23.csv",
        "Gram": "gram_des_1997_98_2022_23.csv",
        "Moong": "moong_des_1997_98_2022_23.csv",
        "Soybean": "soybean_des_1997_98_2022_23.csv",
        "Groundnut": "groundnut_des_1997_98_2022_23.csv",
        "Rapeseed & Mustard": "rapeseed_mustard_des_1997_98_2022_23.csv"
    }

    output_path = PROCESSED_DIR / filename_map[crop]

    df.to_csv(output_path, index=False)

    saved_files[crop] = output_path

    print(f"✓ {crop:22s} -> {output_path.name}")

print(f"TOTAL FILES SAVED: {len(saved_files)}")

SAVING STANDARDIZED CROP DATA
Project root:
D:\Projects\AgriRisk and ROI Prediction\Dump

Output directory:
D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield
✓ Arhar                  -> arhar_des_1997_98_2022_23.csv
✓ Bajra                  -> bajra_des_1997_98_2022_23.csv
✓ Jowar                  -> jowar_des_1997_98_2022_23.csv
✓ Gram                   -> gram_des_1997_98_2022_23.csv
✓ Moong                  -> moong_des_1997_98_2022_23.csv
✓ Soybean                -> soybean_des_1997_98_2022_23.csv
✓ Groundnut              -> groundnut_des_1997_98_2022_23.csv
✓ Rapeseed & Mustard     -> rapeseed_mustard_des_1997_98_2022_23.csv

TOTAL FILES SAVED: 8


In [ ]:
#Verifying saved files
print("VERIFYING SAVED FILES")

for crop, path in saved_files.items():

    exists = path.exists()

    print(f"\n{crop}")
    print(f"Path   : {path}")
    print(f"Exists : {exists}")

    if exists:
        check_df = pd.read_csv(path)

        print(f"Rows   : {len(check_df):,}")
        print(f"Cols   : {len(check_df.columns)}")


VERIFYING SAVED FILES

Arhar
Path   : D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\arhar_des_1997_98_2022_23.csv
Exists : True
Rows   : 87
Cols   : 4

Bajra
Path   : D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\bajra_des_1997_98_2022_23.csv
Exists : True
Rows   : 87
Cols   : 4

Jowar
Path   : D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\jowar_des_1997_98_2022_23.csv
Exists : True
Rows   : 87
Cols   : 4

Gram
Path   : D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\gram_des_1997_98_2022_23.csv
Exists : True
Rows   : 87
Cols   : 4

Moong
Path   : D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\moong_des_1997_98_2022_23.csv
Exists : True
Rows   : 87
Cols   : 4

Soybean
Path   : D:\Projects\AgriRisk and ROI Prediction\Dump\data\processed\crop_yield\soybean_des_1997_98_2022_23.csv
Exists : True
Rows   : 87
Cols   : 4

Groundnut
Path   : D:\Projects\AgriRisk and ROI Pre

In [ ]:
#Final quality check
# Uses the ACTUAL extracted crop data from DES
print("FINAL QUALITY CHECK — NOTEBOOK 05")

#The extracted crop datasets
TARGET_CROPS = {
    "Arhar": ["arhar/tur", "arhar", "tur"],
    "Bajra": ["bajra"],
    "Jowar": ["jowar"],
    "Gram": ["gram"],
    "Moong": ["moong(green gram)", "moong", "green gram"],
    "Soybean": ["soyabean", "soybean"],
    "Groundnut": ["groundnut"],
    "Rapeseed & Mustard": [
        "rapeseed &mustard",
        "rapeseed & mustard",
        "rapeseed",
        "mustard"
    ]
}

#Using the complete DES data
if "des" in globals() and isinstance(des, pd.DataFrame):

    validation_source = des.copy()

elif "des_raw" in globals() and isinstance(des_raw, pd.DataFrame):

    validation_source = des_raw.copy()

else:

    raise RuntimeError(
        "Neither `des` nor `des_raw` is available. "
        "Please run the DES loading cell first."
    )


print("\nDES validation source:")
print("Shape:", validation_source.shape)
print("Columns:", list(validation_source.columns))

#Normalizing crop names
validation_source["_crop_normalized"] = (
    validation_source["crop_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

#Extracting all 8 target crops
standardized_crop_dfs = {}

for target_crop, aliases in TARGET_CROPS.items():

    mask = validation_source["_crop_normalized"].isin(
        [x.lower() for x in aliases]
    )

    temp_crop = validation_source.loc[mask].copy()

    if len(temp_crop) == 0:
        print(f"✗ {target_crop:22s} -> NOT FOUND")
        continue

    # Adding standardized crop name
    temp_crop["crop"] = target_crop

    # Renaming DES fields to standardized names
    rename_map = {
        "state_name": "state",
        "district_name": "district",
        "area": "area_ha",
        "production": "production_tonnes",
        "yield": "yield_reported",
        "crop_name": "crop_source_name"
    }

    temp_crop = temp_crop.rename(columns=rename_map)

    standardized_crop_dfs[target_crop] = temp_crop

    print(
        f"✓ {target_crop:22s} -> "
        f"{len(temp_crop):,} rows"
    )

#Verifying all crops
print("EXTRACTED CROP DATAFRAMES")

for crop in TARGET_CROPS:

    if crop in standardized_crop_dfs:

        df_crop = standardized_crop_dfs[crop]

        print(
            f"✓ {crop:22s} "
            f"Rows: {len(df_crop):7,d} | "
            f"States: {df_crop['state'].nunique():3d} | "
            f"Districts: {df_crop['district'].nunique():3d}"
        )

    else:

        print(f"✗ {crop:22s} MISSING")

#Combine all 8 crops
if len(standardized_crop_dfs) != len(TARGET_CROPS):

    missing = [
        crop
        for crop in TARGET_CROPS
        if crop not in standardized_crop_dfs
    ]

    raise RuntimeError(
        f"The following target crops are missing: {missing}"
    )


remaining_crops_df = pd.concat(
    list(standardized_crop_dfs.values()),
    ignore_index=True
)


print("COMBINED DATASET")

print("Rows:", len(remaining_crops_df))
print("Columns:", len(remaining_crops_df.columns))
print("Crops:", remaining_crops_df["crop"].nunique())
print("States:", remaining_crops_df["state"].nunique())
print("Districts:", remaining_crops_df["district"].nunique())

#Missing value check
print("MISSING VALUE CHECK")

important_columns = [
    "year",
    "state",
    "district",
    "crop",
    "area_ha",
    "production_tonnes",
    "yield_reported"
]

missing_rows = []

for col in important_columns:

    missing_count = remaining_crops_df[col].isna().sum()

    missing_rows.append({
        "column": col,
        "missing_count": missing_count,
        "missing_pct": (
            missing_count / len(remaining_crops_df) * 100
        )
    })

missing_summary = pd.DataFrame(missing_rows)

display(missing_summary)

#Zero-value check
print("ZERO VALUE CHECK")

zero_rows = []

for col in [
    "area_ha",
    "production_tonnes",
    "yield_reported"
]:

    numeric_values = pd.to_numeric(
        remaining_crops_df[col],
        errors="coerce"
    )

    zero_count = (numeric_values == 0).sum()

    zero_rows.append({
        "column": col,
        "zero_count": zero_count,
        "zero_pct": (
            zero_count / len(remaining_crops_df) * 100
        )
    })

zero_summary = pd.DataFrame(zero_rows)

display(zero_summary)

#Duplicate check
print("DUPLICATE CHECK")

duplicate_keys = [
    "year",
    "state",
    "district",
    "crop",
    "season"
]

duplicate_count = remaining_crops_df.duplicated(
    subset=duplicate_keys
).sum()

print("Duplicate key columns:", duplicate_keys)
print("Duplicate rows:", duplicate_count)

#Crop-wise final summary
print("FINAL CROP-WISE SUMMARY")

final_crop_summary = (
    remaining_crops_df
    .groupby("crop")
    .agg(
        rows=("crop", "size"),
        states=("state", "nunique"),
        districts=("district", "nunique"),
        years=("year", "nunique")
    )
    .reset_index()
)

display(final_crop_summary)

#Year coverage
print("YEAR COVERAGE")

year_summary = (
    remaining_crops_df
    .groupby("crop")["year"]
    .agg(["min", "max", "nunique"])
    .reset_index()
    .rename(columns={
        "min": "min_year",
        "max": "max_year",
        "nunique": "number_of_years"
    })
)

display(year_summary)

#Final target-crop completeness
print("TARGET CROP COMPLETENESS")

found_crops = set(
    remaining_crops_df["crop"].unique()
)

missing_crops = []

for crop in TARGET_CROPS:

    if crop in found_crops:
        print(f"✓ {crop}")

    else:
        print(f"✗ {crop}")
        missing_crops.append(crop)

#Final verdict
print("NOTEBOOK 05 FINAL VERDICT")

if len(missing_crops) == 0:

    print("✓ ALL 8 REMAINING CROPS ARE PRESENT")

else:

    print("✗ Missing crops:", missing_crops)


if duplicate_count == 0:

    print("✓ No duplicate year-state-district-crop-season records")

else:

    print(
        f"⚠ {duplicate_count} duplicate "
        "year-state-district-crop-season records found"
    )


print("\nFinal combined dataset:")
print("Rows       :", len(remaining_crops_df))
print("Crops      :", remaining_crops_df["crop"].nunique())
print("States     :", remaining_crops_df["state"].nunique())
print("Districts  :", remaining_crops_df["district"].nunique())

print("END OF NOTEBOOK 05 QUALITY CHECK")

FINAL QUALITY CHECK — NOTEBOOK 05

DES validation source:
Shape: (455359, 18)
Columns: ['_id', 'id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit', '_crop_name_normalized']
✓ Arhar                  -> 13,230 rows
✓ Bajra                  -> 10,376 rows
✓ Jowar                  -> 13,841 rows
✓ Gram                   -> 12,119 rows
✓ Moong                  -> 23,419 rows
✓ Soybean                -> 6,160 rows
✓ Groundnut              -> 18,623 rows
✓ Rapeseed & Mustard     -> 12,661 rows

EXTRACTED CROP DATAFRAMES
✓ Arhar                  Rows:  13,230 | States:  30 | Districts: 669
✓ Bajra                  Rows:  10,376 | States:  24 | Districts: 517
✓ Jowar                  Rows:  13,841 | States:  24 | Districts: 532
✓ Gram                   Rows:  12,119 | States:  29 | Districts: 643
✓ Moong                  Rows:  23,419 | States:  30

,column,missing_count,missing_pct
0,year,0,0.000000
1,state,0,0.000000
2,district,0,0.000000
3,crop,0,0.000000
4,area_ha,0,0.000000
5,production_tonnes,918,0.831303
6,yield_reported,0,0.000000



ZERO VALUE CHECK


,column,zero_count,zero_pct
0,area_ha,0,0.000000
1,production_tonnes,327,0.296118
2,yield_reported,1245,1.127421



DUPLICATE CHECK
Duplicate key columns: ['year', 'state', 'district', 'crop', 'season']
Duplicate rows: 0

FINAL CROP-WISE SUMMARY


,crop,rows,states,districts,years
0,Arhar,13230,30,669,26
1,Bajra,10376,24,517,26
2,Gram,12119,29,643,26
3,Groundnut,18623,29,600,26
4,Jowar,13841,24,532,26
5,Moong,23419,30,685,26
6,Rapeseed & Mustard,12661,33,662,26
7,Soybean,6160,25,452,26



YEAR COVERAGE


,crop,min_year,max_year,number_of_years
0,Arhar,1997-1998,2022-2023,26
1,Bajra,1997-1998,2022-2023,26
2,Gram,1997-1998,2022-2023,26
3,Groundnut,1997-1998,2022-2023,26
4,Jowar,1997-1998,2022-2023,26
5,Moong,1997-1998,2022-2023,26
6,Rapeseed & Mustard,1997-1998,2022-2023,26
7,Soybean,1997-1998,2022-2023,26



TARGET CROP COMPLETENESS
✓ Arhar
✓ Bajra
✓ Jowar
✓ Gram
✓ Moong
✓ Soybean
✓ Groundnut
✓ Rapeseed & Mustard

NOTEBOOK 05 FINAL VERDICT
✓ ALL 8 REMAINING CROPS ARE PRESENT
✓ No duplicate year-state-district-crop-season records

Final combined dataset:
Rows       : 110429
Crops      : 8
States     : 34
Districts  : 732

END OF NOTEBOOK 05 QUALITY CHECK
